# Building an MDMC Universe
An MDMC simulation requires a configuration and a topology defined within a Universe object. Although MDMC is somewhat flexible with regards to the order in which these are created, a suggested approach is as follows:
## Create the Universe
MDMC currently only supports orthorhombic boxed Universes; the dimensions must be specified when creating (initialising) the Universe object:

In [1]:
# Import the Universe class
from MDMC.MD.simulation import Universe
# Initialise a Universe with dimensions in Ang
universe = Universe([10.0, 15.0, 20.0])

ImportError: No module named MDMC.MD.simulation

To create a cubic Universe, only a single float needs to be specified for the dimensions:

In [2]:
universe = Universe(10.0)

NameError: name 'Universe' is not defined

## Create an atomic configuration
Currently configurations must be specified by the user; in the future users will also be able to provide a pdb/cif/xyz file to create the atomic configuration.

### Create an atom
Each atom is specified using an Atom object, which possesses (amongst other things) a position, velocity, elemental symbol, mass, charge, and atom type.  At a minimum the elemental symbol must be specified when creating an Atom object:

In [3]:
# Import the Atom class
from MDMC.MD.structural_units import Atom
# Create a hydrogen atom
H1 = Atom('H')

ImportError: No module named MDMC.MD.structural_units

This will create a Hydrogen atom with mass determined from an elemental lookup table, position and velocity of (0., 0., 0.) and no charge or atom type.  Atoms can also be created by copying another atom and passing the position of the new atom:

In [4]:
H2 = H1.copy(position=[1., 1., 1.])

NameError: name 'H1' is not defined

The copied atom will have identical properties to the original attribute, except with a different ID (which is unique for all structural units) and a different position.  This includes interactions, which will applying to the copied atom in the same way as the original atom e.g. if H1 is bonded to an atom O, H2 will also be bonded to atom O.

### Atom type
Each atom has an atom_type, which is used for applying non-bonded interactions between atoms; all atoms of the same atom_type will have the same non-bonded interactions applying to them.  The atom_type can be specified when the atom is created:

In [7]:
O1 = Atom('O', position=[8.0, 8.0, 8.0], atom_type=1)
# and to add the created atom to universe do
universe.add_structural_unit(O1)

NameError: name 'Atom' is not defined

Alternatively, the atom_type can be inferred by MDMC.  This occurs when an atom with no defined atom_type is added to a Universe:

In [6]:
O2 = Atom('O', position=[9.0, 9.0, 9.0])
universe.add_structural_unit(O2)

NameError: name 'Atom' is not defined

Assuming no other atoms have been added to the universe, this sets ```O2.atom_type``` equal to 1.  MDMC infers the atom_type of each atom based on its element and its interactions: all atoms with the same element and the same interactions **when they are added to the universe** will have the same atom_type.  **Once an atom's atom_type is set, it is immutable i.e. it cannot be changed.**
It is recommended that either all atom types are specified when atoms are created or none are (i.e. either MDMC is left to infer and assign all atom types or no atom types.  Only specifying some atom types could result in unexpected behaviour; if it is absolutely necessary then ensure that all atoms with specified atom types are added to the universe first).

To see what atoms have been added to the universe:

In [8]:
universe.structure_list

NameError: name 'universe' is not defined

You can see other attributes of the universe (or any other MDMC containers) with:

In [9]:
help(universe)

NameError: name 'universe' is not defined

The same applies for methods, for example:

In [10]:
help(universe.add_structural_unit)

NameError: name 'universe' is not defined

### Create bonded interactions
Currently two bonded interaction types exist within MDMC: Bond and BondAngle.  Each interaction must have an interaction function which describes the interaction. Below this is demonstrated where a bond with a harmonic potential function is created.

In [11]:
# Import Bond and HarmonicPotential
from MDMC.MD.structural_units import Bond
from MDMC.MD.interaction_functions import HarmonicPotential
# Create a Bond with a HarmonicPotential
# The first argument in the HarmonicPotential is the equilibrium state and the second is the potential strength
# Currently the units of the equilibrium state and potential strength parameters must be provided
HH_bond = Bond(H1, H2, function=HarmonicPotential((1., 'Ang'), (100., 'kJ / mol Ang^2')))

ImportError: No module named MDMC.MD.structural_units

To see which units are supported:

In [12]:
from MDMC.common.units import SYSTEM
SYSTEM

ImportError: No module named MDMC.common.units

A BondAngle is created in the same manner except it requires a minimum of three atoms.  Currently HarmonicPotential is the only bonded interaction function that exists, and it can be applied to either Bond or BondAngle.
All bonds and bond angles can be constrained; this can either be set when creating the bond or afterwards:

In [13]:
HH_bond.constrained = True

NameError: name 'HH_bond' is not defined

For a constraint to applied during a simulation, the universe must have a constraint algorithm.

### Create a molecule
A molecule consists of two or more atoms and at least one bonded interaction:

In [15]:
# Import the Atom, Molecule, Bond, BondAngle and HarmonicPotential classes
from MDMC.MD.structural_units import Molecule, BondAngle
# Create a H2 molecule
H_mol = Molecule(position=[2.0, 2.0, 2.0], atoms=[H1, H2], interactions=[HH_bond])

ImportError: No module named MDMC.MD.structural_units

When a molecule is created, the position of the atoms relative to one another is fixed.  The atoms are then moved so that the position of the molecular centre of mass is what was passed when creating the molecule.  In the example above, the atoms were at [0., 0., 0.] and [1., 1., 1.] before the molecule was created; therefore they will always be separated by 1.0 Ang in each dimension, no matter where the molecule is moved to.  The molecular centre of mass is set to [1.5, 1.5, 1.5], so the atom positions are changed to [1.5, 1.5, 1.5] and [2.5, 2.5, 2.5] respectively.
It is also possible to copy molecules:

In [16]:
H_mol2 = H_mol.copy(position=[5., 5., 5.])

NameError: name 'H_mol' is not defined

When a molecule is copied, each atom is copied, as are all of the bonded interactions between these atoms (and all of the non-bonded interactions).

One method for building molecules is to copy atoms, as the interactions are also copied. For example, to build methane:

In [17]:
m_C = Atom('C')
# Define the H atom positions relative to a C at [0,0,0]
d = 0.629118
H_pos = [[d, d, d], [-d, -d, d], [d, -d, -d], [-d, d, -d]]
m_H = Atom('H', position=H_pos[0])
# Add a bond between 
CH_bonds = Bond(m_C, m_H, function=HarmonicPotential((1.09, 'Ang'), (100., 'kJ / mol Ang^2')))
# Make three H atom copies (which are therefore all bonded to m_C)
H_atoms = [m_H]
for pos in H_pos[1:]:
    H_atoms.append(m_H.copy(pos))
# The next two lines simply create a list of unique HCH triplets
# e.g. [(m_H1, m_C, m_H2), (m_H1, m_C, m_H3), ..., (m_H3, m_C, m_H4)].
from itertools import combinations
HCH_triplets = [(i[0], m_C, i[1]) for i in combinations(H_atoms, 2)]
# Unpack the list of triplets with * notation i.e. [(...), (...), (...)] becomes (...), (...), (...)
HCH_angles = BondAngle(*HCH_triplets, function=HarmonicPotential((109.5, 'deg'), (10., 'kJ / mol deg^2')))
# Create a methane molecule by adding C atom to list of H atoms
methane = Molecule(atoms=[m_C]+H_atoms)

NameError: name 'Atom' is not defined

### Add structural units to a universe
There are two methods for adding a structural unit to a universe:

In [18]:
# Add an individual structural unit to the universe
universe.add_structural_unit(H_mol)
# Fill the universe with the structural unit repeated on a cubic lattice with a specific number density
universe.fill(H_mol, num_density = 0.01)

NameError: name 'universe' is not defined

Currently the fill command is limited to filling with a cubic lattice, and cannot be used in conjunction with add_structural_unit.

### Create non-bonded interactions
Non-bonded interactions are applied to atoms based on their atom_type, rather than to individual atoms.  They must also have a universe specified, so that they know which atoms they apply to (atoms must have an atom_type once they have been added to a universe).  For example, to create a Dispersion interaction with a Lennard-Jones interaction function between two atom types:

In [19]:
# Import the Dispersion interaction and Lennard-Jones function
from MDMC.MD.structural_units import Dispersion
from MDMC.MD.interaction_functions import LennardJones
# Create a dispersion interaction with a Lennard-Jones function between atoms with atom_type 1 (O) and atom_type 2 (H)
# The first LJ parameter is epsilon and the second is sigma
# The cutoff for the dispersion interaction can also be set (in Ang)
LJ_HO = Dispersion(universe, [1, 2], function=LennardJones((0.65, 'kJ / mol'), (3., 'Ang')), cutoff=10.)

ImportError: No module named MDMC.MD.structural_units

The exception to this is Coulombic interactions, which can be applied either to a list of atoms or to a list of atom types.  If the Coulombic interaction is applied to a list of atoms, the universe does not need to be specified:

In [20]:
# Import the Coulombic interaction
from MDMC.MD.structural_units import Coulombic
# Create a Coulombic interaction with a Coulomb potential and a charge of 0.42
c_H = Coulombic(atoms=[H1, H2], charge=0.42)

ImportError: No module named MDMC.MD.structural_units

As Coulombic interactions typically have the same interaction function (i.e. a Coulomb function, where the force results in Coulomb's law), Coulombic interactions do not need to be specified with an interaction function (although other functions can be provided); to set the function as Coulomb, a value for the charge can be passed.  This is equivalent to:

In [21]:
from MDMC.MD.interaction_functions import Coulomb
c_H = Coulombic(atoms=[H1, H2], function=Coulomb((0.42, 'e')))

ImportError: No module named MDMC.MD.interaction_functions

As with all non-bonded interactions, a Coulombic interaction can also be created by specifying the atom types:

In [22]:
c_O = Coulombic(universe, 1, charge=0.42)

NameError: name 'Coulombic' is not defined

In this case a universe must be provided.

### Adding kspace solvers to a universe
There are several solvers for determining the long range energy contribution for non-bonded interactions, including Ewald, particle-particle particle-mesh (PPPM), and particle-mesh Ewald (PME).  If the long range contribution to the non-bonded energy (i.e. > cutoff) is to be calculated during a simulation, a kspace solver has to be added to the universe.  A solver can be specified for either electrostatic or dispersive interactions, or both. This can either be during universe initialisation or afterwards:

In [23]:
# Import Ewald and PPPM kspace solvers
from MDMC.MD.simulation import Ewald, PPPM
# Create an ewald solver
ewald = Ewald(accuracy=1e-5)
# Initialise a universe with an Ewald solver for both electrostatics and dispersive interactions
uni1 = Universe(10., kspace_solver=ewald)
# Initialise a universe and then add a PPPM solver for electrostatic interactions
uni2 = Universe(10.)
pppm = PPPM(accuracy=1e-4)
uni2.electrostatic_solver = pppm
# Initialise a universe with a PPPM solver for dispersive interactions
uni3 = Universe(10., dispersive_solver=PPPM)

ImportError: No module named MDMC.MD.simulation

Not all kspace solvers are implemented for all MD engines, and they may also require different parameters to be specified - see the MD engine documentation for more information.

### Adding constraint algorithms to a universe
In a similar manner to kspace solvers, constraint algorithms (e.g. SHAKE, RATTLE) can also be passed to a universe.  A constraint algorithm is required if any of the bonded interactions are constrained.

In [24]:
# Import Shake and Rattle
from MDMC.MD.simulation import Shake, Rattle
# Initialise a universe with the Shake algorithm
# The first Shake parameter is the accuracy and the second is the maximum number of iterations used for any constraint calculation
shake = Shake(1e-4, 100)
uni4 = Universe(10., constraint_algorithm=shake)
# Add Rattle after universe initialisation
rattle = Rattle(1e-5, 1000)
uni5 = Universe(10.)
uni5.constraint_algorithm = rattle

ImportError: No module named MDMC.MD.simulation

Not all constraint algorithms are implemented for all MD engines, and they may also required different parameters to be specified - see the MD engine documentation for more information.

## Example Universe filled with SPCE water
For water, SPCE and SPC forcefields are predefined, so parameters do not need to be set when interactions are defined; instead parameters are set by adding a forcefield to the universe.

In [25]:
from MDMC.MD.simulation import Universe, Shake, PPPM
from MDMC.MD.structural_units import Atom, Bond, BondAngle, Coulombic, Dispersion, Molecule

universe = Universe(dimensions=21.75, constraint_algorithm=Shake(1e-4, 100), electrostatic_solver=PPPM(accuracy=1e-5))
H1 = Atom('H')
H2 = Atom('H', position=(0., 1.63298, 0.))
O = Atom('O', position=(0., 0.81649, 0.57736))
H_coulombic = Coulombic(atoms=[H1, H2], cutoff=10.)
O_coulombic = Coulombic(atoms=O, cutoff=10.)
water_mol = Molecule(position=(0, 0, 0),
                     velocity=(0, 0, 0),
                     atoms=[H1, H2, O],
                     interactions=[Bond((H1, O), (H2, O), constrained=True),
                                   BondAngle(H1, O, H2, constrained=True)],
                     name='water')
universe.fill(water_mol, num_density=0.03356718472021752)
O_dispersion = Dispersion(universe, O.atom_type, cutoff=10., vdw_tail_correction=True)
universe.add_force_field('SPCE')

ImportError: No module named MDMC.MD.simulation